# Object Detection

Build a model that answers two questions at once:
1. **What** is in the image? → Classification
2. **Where** is it? → Bounding Box

We detect 3 shapes: **Circle (0)**, **Rectangle (1)**, **Triangle (2)**

> No GPU needed — runs fine on CPU.

In [ ]:
# Install required packages (run this once)
!pip install torch torchvision numpy matplotlib opencv-python scikit-learn tqdm

In [ ]:
import numpy as np
import cv2
import random
import torch
import torch.nn as nn # neural network module it contains conv2d, relu, pooling,
import matplotlib.pyplot as plt
import matplotlib.patches as patches # used for drawing bounding boxes

from torchvision.transforms import ToTensor  # image - tensor
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split


In [ ]:
IMG_SIZE = 128
CLASS_NAMES = ['Circle', 'Rectangle', 'Triangle']

device = 'cuda' if torch.cuda.is_available() else 'cpu'
to_tensor = ToTensor()

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print('Device:', device)

## Step 1 — Create the Dataset

Each image has one green shape on a black background.  
The bbox `[x, y, w, h]` is **normalized to [0, 1]**.

In [ ]:
def create_sample():

    img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

    cls  = random.randint(0, 2)
    size = random.randint(20, 40)
    cx   = random.randint(size, IMG_SIZE - size)
    cy   = random.randint(size, IMG_SIZE - size)
    color = (0, 255, 0)
    x1 = cx - size
    y1 = cy - size

    if cls == 0:    # Circle
        cv2.circle(img, (cx, cy), size, color, -1)
    elif cls == 1:  # Rectangle
        cv2.rectangle(img, (x1, y1), (cx+size, cy+size), color, -1)
    else:           # Triangle
        pts = np.array([[cx, y1], [x1, cy+size], [cx+size, cy+size]])
        cv2.fillPoly(img, [pts], color)

    bbox = [x1/IMG_SIZE, y1/IMG_SIZE, (2*size)/IMG_SIZE, (2*size)/IMG_SIZE]
    return img, cls, bbox


images, labels, bboxes = [], [], []

for _ in range(1000):
    img, cls, bbox = create_sample()
    images.append(to_tensor(img))
    labels.append(cls)
    bboxes.append(bbox)

X       = torch.stack(images)
y_class = torch.tensor(labels)
y_bbox  = torch.tensor(bboxes, dtype=torch.float32)

print(f'Dataset: {len(images)} samples')

## Step 2 — Visualize Samples


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))

for ax, img_t, cls, bbox in zip(axes.flat, images[:8], labels[:8], bboxes[:8]):

    img_np = np.moveaxis(img_t.numpy(), 0, -1)   # (C,H,W) → (H,W,C) for display
    ax.imshow(img_np)

    x, y, w, h = [v * IMG_SIZE for v in bbox]
    ax.add_patch(patches.Rectangle((x, y), w, h, edgecolor='red', facecolor='none', linewidth=2))

    ax.set_title(CLASS_NAMES[cls])
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 3 — Prepare DataLoaders

Split data 80/20, wrap in `TensorDataset`, then batch with `DataLoader`.

In [ ]:
X_train, X_test, y_train_cls, y_test_cls, y_train_bb, y_test_bb = train_test_split(
    X, y_class, y_bbox,
    test_size=0.2,
    random_state=42
)

train_data = TensorDataset(X_train, y_train_cls, y_train_bb)
test_data  = TensorDataset(X_test,  y_test_cls,  y_test_bb)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=32)

print(f'Train: {len(train_data)}  |  Test: {len(test_data)}')

## Step 4 — Build the Model

```
Image (3×128×128)
   → Conv→ReLU→MaxPool  (×2)
   → Flatten
   → FC (128 units)
   ├── class_head → 3 class scores
   └── box_head   → [x, y, w, h]
```

In [ ]:
class Detector(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten()
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 32 * 32, 128),
            nn.ReLU()
        )

        self.class_head = nn.Linear(128, 3)
        self.box_head   = nn.Linear(128, 4)

    def forward(self, x):
        x = self.features(x)
        x = self.fc(x)
        return self.class_head(x), torch.sigmoid(self.box_head(x))


model = Detector().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Step 5 — Loss & Optimizer

- `CrossEntropyLoss` — for classification
- `MSELoss` — for bounding box coordinates

In [ ]:
class_loss = nn.CrossEntropyLoss()
bbox_loss  = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Step 6 — Train the Model

In [ ]:
EPOCHS = 20

for epoch in range(1, EPOCHS + 1):

    model.train()

    for imgs, cls, bbox in train_loader:

        imgs = imgs.to(device)
        cls  = cls.to(device)
        bbox = bbox.to(device)

        pred_cls, pred_bbox = model(imgs)

        loss = class_loss(pred_cls, cls) + 5 * bbox_loss(pred_bbox, bbox)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 5 == 0:
        print(f'Epoch {epoch}/{EPOCHS}  Loss: {loss.item():.4f}')

print('Training done!')

## Step 7 — Predict on a New Image

Green box = ground truth &nbsp;|&nbsp; Red box = model prediction

In [ ]:
img, true_cls, true_box = create_sample()

x = to_tensor(img).unsqueeze(0).to(device)

model.eval()

with torch.no_grad():
    pred_cls, pred_box = model(x)

pred_cls = pred_cls.argmax(1).item()
pred_box = pred_box.squeeze().cpu().numpy()

fig, ax = plt.subplots()

ax.imshow(img)

# Predicted box (red)
px, py, pw, ph = [v * IMG_SIZE for v in pred_box]
ax.add_patch(patches.Rectangle((px, py), pw, ph, edgecolor='red', facecolor='none', linewidth=2))

# Ground truth box (green)
gx, gy, gw, gh = [v * IMG_SIZE for v in true_box]
ax.add_patch(patches.Rectangle((gx, gy), gw, gh, edgecolor='lime', facecolor='none', linewidth=2))

ax.set_title(f'Predicted: {CLASS_NAMES[pred_cls]}  |  Truth: {CLASS_NAMES[true_cls]}')
ax.axis('off')
plt.show()

## What You Built

| Concept | What You Did |
|---------|-------------|
| Dataset | Synthetic shapes with normalized bounding boxes |
| CNN Backbone | 3× (Conv → ReLU → MaxPool) + Global Avg Pool |
| Multi-task Loss | CrossEntropy (class) + SmoothL1 (bbox) |
| IoU Metric | Measures bounding box overlap quality |

## Next Steps
1. **Multi-object** — detect more than one shape per image (YOLO-style grid)
2. **Real data** — try Pascal VOC with `torchvision.datasets.VOCDetection`
3. **Better backbone** — swap in a pretrained ResNet (transfer learning)
4. **Data augmentation** — flips, crops, color jitter